# Lab 3 · Evaluation — Kaggle T4

## What we're doing

Stop saying *"the fine-tune feels better"* and **put numbers on it**. We score the
base model and the fine-tuned adapter on the **same held-out test set** (the
order → JSON task), then run a standard academic **benchmark** for contrast. A
fine-tune can lift *your* task metric a lot while barely moving a general
benchmark — trust the held-out task eval for your job.

## Self-contained

Ideally this reuses **Lab 2's** adapter (Add Input → `aibase-lab2-finetune`). But
if that's not attached, the notebook **regenerates the seeded dataset and trains
its own adapter inline** — so it always runs. Either way you get the same
base-vs-fine-tuned comparison.

## What you'll see

- **Part A — task eval.** A base-vs-fine-tuned table on `json_valid_%`,
  `exact_match_%`, and per-field accuracy (`intent`/`item`/`qty`/`city`). Expect
  **exact-match** and the convention fields (`qty`/`city`) to jump.
- **Part B — benchmark.** `arc_easy` via lm-eval — general ability the fine-tune
  shouldn't change much.

---

**Run from the Kaggle editor on a T4** (Accelerator = *GPU T4 x2*, Internet =
*On*), then **Save Version → Save & Run All**.

## Cell 1 — single GPU + deps + look for Lab 2's adapter

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 'GPU T4 x2' = two T4s; pin to one (set BEFORE torch)

import torch
open("/tmp/constraints.txt", "w").write("torch==" + torch.__version__.split("+")[0] + "\n")
!pip -q install -c /tmp/constraints.txt trl peft bitsandbytes datasets accelerate lm-eval

import glob
print("torch:", torch.__version__, "| capability sm_%d%d" % torch.cuda.get_device_capability())

print("\ncontents of /kaggle/input:")
mounted = glob.glob("/kaggle/input/*")
for p in mounted:
    print("  ", p, "->", os.listdir(p)[:6])
if not mounted:
    print("   (nothing mounted — will train an adapter inline)")

# Prefer Lab 2's attached output; else stay None and Cell 2 builds it inline.
cands = glob.glob("/kaggle/input/*/adapters/lora") + glob.glob("/kaggle/input/*/*/adapters/lora")
ADAPTER = next((p for p in cands if os.path.isdir(p)), None)
tcands = glob.glob("/kaggle/input/*/data/test.jsonl") + glob.glob("/kaggle/input/*/*/data/test.jsonl")
TEST = next((p for p in tcands if os.path.isfile(p)), None)
print("\nattached adapter:", ADAPTER, "| attached test:", TEST)

## Cell 2 — if no adapter is attached, build one inline (seeded data → QLoRA)

This mirrors Lab 2 exactly (same seeded dataset, same QLoRA recipe). It only runs
when Lab 2's output isn't attached as an input — so Lab 3 is never blocked.

In [ ]:
import json, random, gc
MODEL = "Qwen/Qwen2.5-3B-Instruct"

if ADAPTER is None:
    print("No Lab 2 adapter attached -> generating data + training inline.\n")
    random.seed(0)
    ITEMS = ["lavender shampoo","green tea","running shoes","USB-C cable","yoga mat",
             "coffee beans","phone case","water bottle","notebook","wireless mouse",
             "desk lamp","protein powder"]
    CITIES = ["Hanoi","Ho Chi Minh City","Da Nang","Singapore","Tokyo","Seattle"]
    INTENTS = {"order":["I'd like to order {q} {item}","can I get {q} {item}",
                 "please send me {q} {item}","order {q} {item} for me","buy {q} {item}",
                 "I want {q} {item} shipped to {city}"],
               "cancel":["cancel my order of {item}","I want to cancel the {item}",
                 "please cancel {q} {item}","stop the {item} order"],
               "track":["where is my {item}","track my {item} order",
                 "status of my {item}","has my {item} shipped yet"]}
    SYSTEM = ('You are an order-intent parser. Reply with ONLY a JSON object with keys '
              '"intent" (order|cancel|track), "item" (string), "qty" (integer), '
              '"city" (string or null). No prose, no code fences.')
    def ex():
        intent=random.choice(list(INTENTS)); item=random.choice(ITEMS)
        q=random.randint(1,5); city=random.choice(CITIES)
        text=random.choice(INTENTS[intent]).format(q=q,item=item,city=city)
        has_city="{city}" in text or random.random()<0.3
        if "{city}" not in text and has_city: text+=f" to {city}"
        tgt={"intent":intent,"item":item,"qty":q if intent!="track" else 1,
             "city":city if has_city else None}
        return {"messages":[{"role":"system","content":SYSTEM},
                            {"role":"user","content":text},
                            {"role":"assistant","content":json.dumps(tgt,ensure_ascii=False)}]}
    os.makedirs("data", exist_ok=True)
    seen, uniq = set(), []
    for _ in range(3000):
        r=ex(); k=r["messages"][1]["content"]
        if k not in seen: seen.add(k); uniq.append(r)
    random.shuffle(uniq); split=int(len(uniq)*0.85)
    for name, part in [("train", uniq[:split]), ("test", uniq[split:])]:
        with open(f"data/{name}.jsonl","w") as f:
            for r in part: f.write(json.dumps(r,ensure_ascii=False)+"\n")
    print(f"data: {split} train / {len(uniq)-split} test")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer
    from datasets import load_dataset
    use_qlora = torch.cuda.get_device_capability()[0] >= 7
    quant = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
             if use_qlora else None)
    tok = AutoTokenizer.from_pretrained(MODEL)
    m = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant,
            dtype=torch.float16, device_map="auto")
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    ds = load_dataset("json", data_files={"train":"data/train.jsonl"})["train"]
    cfg = SFTConfig(output_dir="adapters/lora", num_train_epochs=2,
            per_device_train_batch_size=2, gradient_accumulation_steps=8,
            learning_rate=2e-4, warmup_ratio=0.03, logging_steps=20,
            fp16=False, bf16=False, max_length=512,         # no GradScaler (BFloat16 unscale bug)
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}, report_to="none")
    trainer = SFTTrainer(model=m, args=cfg, train_dataset=ds, peft_config=lora)
    trainer.train()
    trainer.save_model("adapters/lora"); tok.save_pretrained("adapters/lora")  # saves the LoRA adapter
    print(f"trained inline | peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    del trainer, m; gc.collect(); torch.cuda.empty_cache()
    ADAPTER, TEST = "adapters/lora", "data/test.jsonl"
else:
    print("Using attached Lab 2 adapter:", ADAPTER)
print("ADAPTER =", ADAPTER, "| TEST =", TEST)

## Part A — held-out task eval (base vs fine-tuned)

For every test example we feed the **system + user** turns to each model, parse
the reply, and compare to the gold JSON. Three metrics: **JSON-valid %**,
**exact-match %**, and per-field accuracy (intent / item / qty / city).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-3B-Instruct"
FIELDS = ["intent", "item", "qty", "city"]
tests = [json.loads(l) for l in open(TEST)]
print(f"base: {BASE}  test set: {len(tests)} examples")

def gen(tok, model, messages):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=128, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def try_parse(text):
    t = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try: return json.loads(t)
    except Exception: return None

def score(tok, model):
    valid = exact = 0; hits = {f: 0 for f in FIELDS}
    for t in tests:
        gold = json.loads(t["messages"][2]["content"])
        pred = try_parse(gen(tok, model, t["messages"][:2]))
        if pred is None: continue
        valid += 1; exact += (pred == gold)
        for f in FIELDS:
            if pred.get(f) == gold.get(f): hits[f] += 1
    n = len(tests)
    return {"json_valid_%": 100*valid/n, "exact_match_%": 100*exact/n,
            **{f"{f}_acc_%": 100*hits[f]/n for f in FIELDS}}

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto").eval()  # fp16
base_m = score(tok, base); del base; gc.collect(); torch.cuda.empty_cache()
ft = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto"), ADAPTER).eval()
ft_m = score(tok, ft)

print(f"\n{'metric':<16}{'BASE':>10}{'FINE-TUNED':>14}{'Δ':>10}")
print("-" * 50)
for k in base_m:
    print(f"{k:<16}{base_m[k]:>9.1f}%{ft_m[k]:>13.1f}%{ft_m[k]-base_m[k]:>+9.1f}")
del ft; gc.collect(); torch.cuda.empty_cache()

## Part B — standardized benchmark (lm-eval-harness)

`arc_easy`, capped at 200 examples for speed. A leaderboard number measures
*general* ability; a fine-tune can lift the task metric above while barely moving
this. Slow on the free GPU — skip if you're short on quota.

In [ ]:
!lm_eval --model hf \
  --model_args pretrained=Qwen/Qwen2.5-3B-Instruct,dtype=float16 \
  --tasks arc_easy --device cuda:0 --batch_size auto --limit 200

## Teaching points

- **Contamination:** a high benchmark score can be memorized test data. Your own
  held-out task eval (Part A) is harder to game — trust it more.
- **One number lies:** report quality **and** speed **and** cost together (tie
  back to Lab 1's tokens/s and VRAM).
- **Eval gates CI:** in MLOps this script becomes a pipeline gate — a regression
  on `exact_match_%` blocks the deploy.
- **Part C (optional):** re-run Part A against a quantized serve (the Lab 1 Ollama
  Q4 endpoint) to measure whether Q4 costs task accuracy. *Measure, don't assume.*